# CELESTE component and system models: a baseline for comparing versions

This notebook builds a fixed set of unit models and system models, solves them, and stores every number it can read
off them (flange state, heat and work, exergy destruction, entropy generation, voltages, ...) in a **snapshot**
(`comparison/snapshots/<label>.json`). The last section compares two snapshots.

**Workflow**

1. Check out a version of CELESTE, run the notebook top to bottom. The snapshot is named after the git branch and
   commit of the `celeste` package that was imported, or after the environment variable `CELESTE_LABEL`.
2. Check out another version, run it again.
3. Set `FILE_A` and `FILE_B` in the last section (any two snapshots) and run the comparison cells.

All models are built with the **former constructor keywords** (`F0_n=`, `F0_T=`, `T=`, `P=`, ...) so that the same
notebook runs on the original code and on the reworked code, where these keywords still work (and are deprecated).
A model that cannot be built or solved by one version is recorded with its error; that is a result, not a failure
of the notebook. Models that exist in only one version (e.g. `MethanolLoop`, `Flash`) are not part of the baseline.

Differences that are intended are listed in the `CHANGELOG.md` section *Results that change*; the comparison shows
everything, so check against that list.

The original code needs `numba` and `cantera` in addition to the usual dependencies.

In [ ]:
import contextlib, io, json, os, re, subprocess, time, traceback, warnings
from pathlib import Path

import numpy as np
import pandas as pd

import celeste
from celeste.CSL import interfaces, media, components, flow, heat, reactors, electrochem
from celeste.CSL.systems import ComponentModels, ElectBox, SOCsystem

warnings.simplefilter('ignore')          # deprecation and convergence warnings: success flags are recorded instead
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80); pd.set_option('display.max_rows', 400)

def _git(*args):
    try:
        out = subprocess.run(['git', '-C', str(Path(celeste.__file__).parent), *args],
                             capture_output=True, text=True, timeout=10)
        return out.stdout.strip()
    except Exception:
        return ''

LABEL = os.environ.get('CELESTE_LABEL') or f"{_git('rev-parse', '--abbrev-ref', 'HEAD') or 'unknown'}@{_git('rev-parse', '--short', 'HEAD')}"
LABEL = re.sub(r'[^A-Za-z0-9_.@-]+', '_', LABEL)
SNAPSHOT_DIR = Path('snapshots') if Path('snapshots').exists() or Path.cwd().name == 'comparison' else Path('comparison/snapshots')
print('celeste imported from', Path(celeste.__file__).parent)
print('snapshot label      ', LABEL)

## Compatibility shim for the original code

The original `SOC.__init__` does not accept the keywords `T_soc0`, `T_soc1`, `P_soc` that the original system models
(`SOECSys`, `SOCsysRecupe`, ...) pass to it, so none of them can be built. If the imported `SOC` lacks these keywords, the
cell below adds them (mapping them to `Tsoc0`, `Tsoc1` and the flange pressures) **for this notebook only**, so that the
system models have a baseline. The reworked code accepts the keywords itself and is not touched. The snapshot records
which shims were applied.

In [ ]:
import inspect
SHIMS = []
if 'T_soc0' not in inspect.signature(electrochem.SOC.__init__).parameters:
    _orig_soc_init = electrochem.SOC.__init__
    def _soc_init(self, *args, T_soc0=None, T_soc1=None, P_soc=None, **kwargs):
        if T_soc0 is not None: kwargs['Tsoc0'] = T_soc0
        if T_soc1 is not None: kwargs['Tsoc1'] = T_soc1
        _orig_soc_init(self, *args, **kwargs)
        if P_soc is not None:
            for fl in (self.F0, self.F1, self.A0, self.A1):
                fl.P = P_soc
    electrochem.SOC.__init__ = _soc_init
    SHIMS.append('SOC accepts T_soc0, T_soc1, P_soc (mapped to Tsoc0, Tsoc1 and the flange pressures)')
print('shims applied:', SHIMS or 'none')

## Harness

`case(group)` runs a function that builds and solves a model and returns it (or a list / dict of models). Everything
numeric that can be read from the model is stored under a flat key `<model>.<port>.<quantity>`: for every flange
`n, T, P, h, s, g, cp, Ex, LHV, m` and the mole fractions `x_<species>`, and for every unit the quantities in `UNIT_PROPS`.
Systems are flattened over their units, plus a few system-level totals. Errors are recorded, never raised.

In [ ]:
FLANGE_PROPS = ('n', 'T', 'P', 'h', 's', 'g', 'cp', 'Ex', 'LHV', 'm')
UNIT_PROPS = ('Q', 'W', 'Wid', 'Exth', 'Exloss', 'Exeff', 'Sgen', 'Vid', 'Vop', 'j', 'RC', 'EnthalpyEfficiency',
              'epsilon', 'conversion', 'xi', 'Qloss')
SYSTEM_CALLS = ('tot_Exloss', 'MBcheck', 'EBcheck', 'SBcheck', 'tot_work_comp', 'tot_Qloss', 'tot_LHV_Flow_in', 'tot_LHV_Flow_out')

RESULTS = {}

def _scalar(v):
    if isinstance(v, (bool, np.bool_)):
        return float(v)
    try:
        v = float(v)
    except Exception:
        return None
    return v if np.isfinite(v) else None

def _put(out, key, v):
    if isinstance(v, dict):
        for k, x in v.items():
            _put(out, f'{key}.{k}', x)
    elif isinstance(v, (list, tuple, np.ndarray)) and np.ndim(v) == 1 and len(v) <= 12:
        for i, x in enumerate(v):
            _put(out, f'{key}[{i}]', x)
    elif (s := _scalar(v)) is not None:
        out[key] = s

def flange_values(fl, prefix):
    out = {}
    for p in FLANGE_PROPS:
        try:
            _put(out, f'{prefix}.{p}', getattr(fl, p))
        except Exception:
            pass
    try:
        for sp, x in zip(fl.Medium.species, fl.Xi):
            out[f'{prefix}.x_{sp}'] = float(x)
    except Exception:
        pass
    return out

def unit_values(unit, prefix, depth=0):
    out = {}
    for k, v in list(vars(unit).items()):
        if isinstance(v, interfaces.Flange):
            out.update(flange_values(v, f'{prefix}.{k}'))
        elif depth < 2 and not k.startswith('_') and isinstance(v, celeste.CSL.ModelBase):
            out.update(unit_values(v, f'{prefix}.{k}' if prefix else k, depth + 1))
    for p in UNIT_PROPS:
        try:
            _put(out, f'{prefix}.{p}', getattr(unit, p))
        except Exception:
            pass
    return out

def run(model, *args, **kwargs):
    """Solve a model and remember the solver result."""
    model._sol = model.run_sol(*args, **kwargs)
    return model

def case(group, name=None):
    def deco(fn):
        key = name or fn.__name__
        rec = dict(group=group, status='ok', values={}, errors={}, success=None, time_s=None)
        t0 = time.perf_counter()
        try:
            with contextlib.redirect_stdout(io.StringIO()):
                out = fn()
            rec['time_s'] = time.perf_counter() - t0
            objs = out if isinstance(out, dict) else (out if isinstance(out, (list, tuple)) else {'': out})
            objs = dict(enumerate(objs)) if isinstance(objs, (list, tuple)) else objs
            succ = []
            for tag, m in objs.items():
                prefix = '' if tag == '' else f'{tag}'
                if isinstance(m, interfaces.Flange):
                    rec['values'].update(flange_values(m, prefix or 'F'))
                    continue
                rec['values'].update(unit_values(m, prefix))
                sol = getattr(m, '_sol', None)
                if sol is not None and hasattr(sol, 'success'):
                    succ.append(bool(sol.success))
                for call in SYSTEM_CALLS:
                    if hasattr(m, call) and isinstance(m, celeste.CSL.SysBase):
                        try:
                            with contextlib.redirect_stdout(io.StringIO()):
                                v = getattr(m, call)()
                            _put(rec['values'], f'{prefix}.{call}' if prefix else call, v)
                        except Exception as e:
                            rec['errors'][call] = f'{type(e).__name__}: {e}'[:200]
                if isinstance(m, celeste.CSL.SysBase):
                    try:
                        succ.extend(bool(v) for v in m.get_success().values())
                    except Exception:
                        pass
            rec['success'] = all(succ) if succ else None
            rec['values'] = {k.lstrip('.'): v for k, v in rec['values'].items()}
        except Exception as e:
            rec['status'] = 'error'
            rec['time_s'] = time.perf_counter() - t0
            rec['errors']['model'] = f'{type(e).__name__}: {e}'[:300]
        RESULTS[key] = rec
        return fn
    return deco

def summary(group=None):
    rows = [dict(case=k, group=r['group'], status=r['status'], solver_ok=r['success'], n_values=len(r['values']),
                 time_s=round(r['time_s'], 3), notes='; '.join(f'{a}: {b}' for a, b in r['errors'].items())[:110])
            for k, r in RESULTS.items() if group is None or r['group'] == group]
    return pd.DataFrame(rows).set_index('case')

In [ ]:
# common feeds
AIR = np.array([0.21, 0.79])
SYNGAS = np.array([0.60, 0.01, 0.20, 0.10, 0.08, 0.01])          # H2, CH4, CO2, CO, H2O, N2 (media.Fuel_CH4)
WET_GAS = np.array([0.30, 0.10, 0.10, 0.10, 0.30, 0.10])
MEOH_FEED = np.array([0.75, 0.05, 0.10, 0.10, 1e-8, 1e-8, 1e-8])  # media.Fuel_MeOH: ..., N2, CH3OH
HB_FEED = np.array([0.74, 0.25, 0.01])
SOC_FEED = np.array([0.05, 1e-8, 0.30, 1e-8, 0.65, 1e-8])          # co-electrolysis: H2, CH4, CO2, CO, H2O, N2

## 1 Media: properties of a stream

`h`, `s`, `g`, `cp`, exergy `Ex` and `LHV` of one mole of every medium at a few states. Reference values for all other
models. States stay inside the fitted temperature range of the species.

In [ ]:
MEDIA_STATES = {
    'Air':        (media.Air,        [(300.0, 1.0), (800.0, 10.0), (1500.0, 1.0)]),
    'Fuel_CH4':   (media.Fuel_CH4,   [(300.0, 1.0), (600.0, 10.0), (1073.15, 1.0)]),
    'Fuel_CH4L':  (media.Fuel_CH4L,  [(300.0, 1.0), (350.0, 5.0)]),
    'Fuel_MeOH':  (media.Fuel_MeOH,  [(400.0, 20.0), (500.0, 50.0)]),
    'Fuel_MeOHL': (media.Fuel_MeOHL, [(300.0, 1.0)]),
    'HB':         (media.HB,         [(500.0, 100.0), (700.0, 200.0)]),
    'Media_AEL':  (media.Media_AEL,  [(300.0, 1.0), (350.0, 5.0)]),
    'Steam':      (media.Steam,      [(400.0, 1.0), (800.0, 10.0)]),
    'Water':      (media.Water,      [(300.0, 1.0), (350.0, 10.0)]),
    'Atm_Air':    (media.Atm_Air,    [(280.0, 1.0)]),
    'media_C3':   (media.media_C3,   [(500.0, 5.0), (900.0, 5.0)]),
}
def _prop_case(label, factory, T, P):
    @case('media', f'props_{label}_{T:g}K_{P:g}bar')
    def _():
        return interfaces.Flange('f', Medium=factory(), n=1.0, T=T, P=P)
for label, (factory, states) in MEDIA_STATES.items():
    for T, P in states:
        _prop_case(label, factory, T, P)
summary('media')

## 2 Flow: mixer and splitter

In [ ]:
@case('flow')
def mixer_air_300_600K():
    m = flow.Mixer('m', media.Air(), F0_n=1.0, F0_Xi=AIR, F1_n=2.0, F1_Xi=AIR)
    m.F0.T, m.F1.T = 300.0, 600.0
    return run(m)

@case('flow')
def mixer_different_compositions():
    m = flow.Mixer('m', media.Fuel_CH4(), F0_n=3.0, F0_Xi=SYNGAS, F1_n=1.0, F1_Xi=WET_GAS)
    m.F0.T, m.F1.T = 450.0, 700.0
    return run(m)

@case('flow')
def splitter_30_percent():
    s = flow.Splitter('s', media.Air(), Split=0.3, F0_n=5.0, F0_Xi=AIR)
    return run(s)
summary('flow')

## 3 Components: compressors, pump, valve, heater, heat exchanger, separators

Valve: the original model leaves the outlet temperature equal to the inlet temperature; the reworked one is isenthalpic
(identical for ideal gases). GPU models: the original `Exloss`/`Sgen` differ by design (see the CHANGELOG).

In [ ]:
@case('components')
def compressor_isentropic_eff1():
    return run(components.CompIsent('c', media.Air(), eff=1.0, F0_n=1.0, F0_Xi=AIR, F0_T=300.0, F0_P=1.0, F1_P=5.0))

@case('components')
def compressor_isentropic_eff08():
    return run(components.CompIsent('c', media.Air(), eff=0.8, F0_n=1.0, F0_Xi=AIR, F0_T=300.0, F0_P=1.0, F1_P=5.0))

@case('components')
def compressor_isentropic_syngas():
    return run(components.CompIsent('c', media.Fuel_CH4(), eff=0.75, F0_n=2.0, F0_Xi=SYNGAS, F0_T=320.0, F0_P=2.0, F1_P=30.0))

@case('components')
def turbine_isentropic():
    return run(components.CompIsent('t', media.Air(), eff=0.85, F0_n=1.0, F0_Xi=AIR, F0_T=900.0, F0_P=8.0, F1_P=1.0))

@case('components')
def compressor_isothermal():
    return run(components.CompIsot('c', media.Air(), eff=0.8, F0_n=1.0, F0_Xi=AIR, F0_T=300.0, F0_P=1.0, F1_P=5.0))

@case('components')
def pump_water():
    return run(components.Pump('p', eff=0.8, F0_T=298.15, F0_P=1.0, F1_P=3.0))

@case('components')
def valve_air_10_to_2bar():
    return run(components.Valve('v', media.Air(), n=1.0, Xi=AIR, F0_T=400.0, F0_P=10.0, F1_P=2.0))

@case('components')
def valve_syngas_10_to_2bar():
    return run(components.Valve('v', media.Fuel_CH4(), n=1.0, Xi=np.array([0.3, 0.1, 0.1, 0.1, 0.3, 0.1]), F0_T=500.0, F0_P=10.0, F1_P=2.0))

@case('components')
def heater_air_heating():
    return run(heat.Heater('h', media.Air(), media.Air(), F0_n=1.0, F0_Xi=np.array([0.6, 0.4]), F0_T=300.0, F1_T=800.0))

@case('components')
def heater_syngas_cooling_to_condensate():
    return run(heat.Heater('h', media.Fuel_CH4(), media.Fuel_CH4L(), F0_n=2.0, F0_Xi=WET_GAS, F0_T=600.0, F0_P=5.0, F1_T=303.15, F1_P=5.0))

@case('components')
def heat_exchanger_ntu_counterflow():
    hx = heat.HexNTU('hx', media.Air(), media.Air(), UA=50.0)
    hx.H0.T, hx.C0.T = 800.0, 300.0
    return run(hx)

@case('components')
def heat_exchanger_effectiveness_09():
    hx = heat.HexNTU('hx', media.Air(), media.Air(), UA=0.9, HexType='effectiveness')
    hx.H0.T, hx.C0.T = 800.0, 300.0
    return run(hx)

@case('components')
def gpu_1s_cooling_separation():
    g = components.GPU_1S('g', media.Fuel_CH4(), media.Fuel_CH4L(), eff=0.7, F0_n=100.0, F0_T=873.15, F0_P=3.0, F1_T=350.0)
    return run(g)

@case('components')
def gpu_2s_remove_water():
    return run(components.GPU_2S('g', media.Fuel_CH4()))

@case('components')
def gpu_2sv2_partial_separation():
    g = components.GPU_2SV2('g', media.Fuel_CH4(), Sep=np.array([1, 1, .5, .4, 0, 1]), F0_Xi=np.array([0.1, 0.3, 0.05, 0.05, 0.5, 1e-8]))
    return run(g)

@case('components')
def gpu_2sv2_efficiency_06():
    g = components.GPU_2SV2('g', media.Fuel_CH4(), Sep=np.array([1, 0, 0.5, 0.2, 0, 0]), eff=0.6,
                           F0_n=3.0, F0_Xi=np.array([0.5, 0.3, 0.1, 0.05, 1e-8, 0.05]), F0_T=400.0, F0_P=5.0)
    return run(g)

@case('components')
def gpu_3s():
    g = components.GPU_3S('g', media.Fuel_CH4(), sep=np.array([1, 0, 1, 1, -1, 1]), F0_Xi=np.array([0.1, 0.1, 1e-8, 1e-8, 0.8, 1e-8]))
    return run(g)

@case('components')
def asu_air_separation():
    return run(components.ASU('a'))

@case('components')
def dac_direct_air_capture():
    return run(components.DAC('d'))
summary('components')

## 4 Reactors

Equilibrium constants come from the NASA Gibbs energies in the reworked code (empirical fits before), so equilibrium
compositions move slightly (within 0.6 % at SOC temperatures, more at low temperature). `MR` below 600 K: the original
solver stopped at points that were not at equilibrium. Methanol reactors, `HB` and `MR` take the state keywords.

In [ ]:
def _mr(T, P, Xi=SYNGAS, n=10.0):
    return run(reactors.MR('r', media.Fuel_CH4(), T=T, P=P, F0_n=n, F0_Xi=Xi))
for _T, _P in [(500.0, 10.0), (600.0, 10.0), (800.0, 5.0), (1000.0, 1.0)]:
    case('reactors', f'MR_methanation_feed_{_T:g}K_{_P:g}bar')(lambda T=_T, P=_P: _mr(T, P))
case('reactors', 'MR_steam_reforming_1100K_1bar')(lambda: _mr(1100.0, 1.0, np.array([0.05, 0.25, 0.01, 0.01, 0.67, 0.01]), 2.0))
case('reactors', 'MR_electrolysis_off_gas_1073K')(lambda: _mr(1073.15, 1.0, SOC_FEED, 1.0))

@case('reactors')
def HB_isothermal():
    return run(reactors.HB('hb', T=673.15, P=200.0, F0_n=1.0, F0_Xi=HB_FEED))

@case('reactors')
def HB_isothermal_550K_100bar():
    return run(reactors.HB('hb', T=550.0, P=100.0, F0_n=1.0, F0_Xi=HB_FEED, Vol=2.0))

@case('reactors')
def HB_adiabatic():
    return run(reactors.HB_Adiabatic('hb', F0_n=1.0, F0_Xi=HB_FEED, F0_T=673.15, F0_P=200.0))

@case('reactors')
def HB_adiabatic_cold_feed():
    return run(reactors.HB_Adiabatic('hb', F0_n=1.0, F0_Xi=HB_FEED, F0_T=620.0, F0_P=150.0, Vol=2.0))

def _meoh(cls, **kw):
    return run(cls('m', media.Fuel_MeOH(), F0_n=1.0, F0_Xi=MEOH_FEED, **kw))
case('reactors', 'MeOH_Equil_473K_50bar')(lambda: _meoh(reactors.MeOH_Equil, T=473.15, P=50.0))
case('reactors', 'MeOH_Equil_523K_80bar')(lambda: _meoh(reactors.MeOH_Equil, T=523.15, P=80.0))
case('reactors', 'MeOH_Yield_473K_50bar')(lambda: _meoh(reactors.MeOH_Yield, T=473.15, P=50.0, Yield=0.35))
case('reactors', 'MeOH_Kinetic_523K_50bar')(lambda: _meoh(reactors.MeOH_Kinetic, T=523.15, P=50.0, M_cat=1000.0))
case('reactors', 'MeOH_WGS_adiabatic_523K_50bar')(lambda: _meoh(reactors.MeOH_WGS, T=523.15, P=50.0, Yield=0.35))

@case('reactors')
def reformer_cstr():
    return run(reactors.Reformer('r', media.media_C3(), F0_Xi=np.array([1e-5, 0.2, 1e-5, 1e-5, 0.7, 1e-5, 0.1]), T=900.0))

@case('reactors')
def reformer_1d_three_stages():
    return run(reactors.Reformer_1D('l', media.media_C3(), F0_Xi=np.array([1e-5, 0.2, 1e-5, 1e-5, 0.7, 1e-5, 0.1]), T=900.0, N=3))
summary('reactors')

## 5 Electrochemical cells

`SOC`: electrolysis (`I < 0`) and fuel cell (`I > 0`). The ideal voltage of the reworked model is the charge-averaged
Nernst potential (`V_model='mean'`); the original one is the Gibbs energy difference of the streams. They agree for an
isothermal stack fed with an equilibrium mixture and differ otherwise. `SOC` with different inlet and outlet temperatures
shows the case where the stream Gibbs energy difference is no longer a voltage.

In [ ]:
def _soc(I, Xi=None, n=1.0, nA=2.0, **kw):
    s = electrochem.SOC('s', media.Fuel_CH4(), media.Air(), I=I, Ncells=100, **kw)
    if Xi is not None:
        s.F0.Xi, s.F0.n, s.A0.n = Xi, n, nA
    return run(s)

case('electrochem', 'SOC_electrolysis_default_feed')(lambda: _soc(-50.0))
case('electrochem', 'SOC_electrolysis_default_feed_300A')(lambda: _soc(-100.0))
case('electrochem', 'SOC_co_electrolysis')(lambda: _soc(-200.0, SOC_FEED))
case('electrochem', 'SOC_co_electrolysis_extent_flag')(lambda: _soc(-200.0, SOC_FEED, extent=True))
case('electrochem', 'SOC_fuel_cell_internal_reforming')(lambda: _soc(300.0, np.array([0.2, 0.25, 0.05, 0.05, 0.45, 1e-8])))
case('electrochem', 'SOC_fuel_cell_hydrogen')(lambda: _soc(50.0, np.array([0.9, 1e-8, 1e-8, 1e-8, 0.1, 1e-8]), n=1.0, nA=3.0))
case('electrochem', 'SOC_electrolysis_1000K_to_1050K')(lambda: _soc(-10.0, Tsoc0=1000.0, Tsoc1=1050.0, P_soc=3.0))
case('electrochem', 'SOC_electrolysis_with_ASR_factor')(lambda: _soc(-50.0, f_ASR=1.5))

@case('electrochem')
def AEL_alkaline_electrolyser():
    return run(electrochem.AEL('a', media.Media_AEL(), Ncells=100, A=800, T=350.0))
summary('electrochem')

## 6 System models

Flowsheets assembled from the units above. Several of these did not run in the original code (see the CHANGELOG, *Fixed*);
their error is recorded. Totals (`tot_Exloss`, `MBcheck`, `EBcheck`, ...) are stored when the system provides them;
`MBcheck` has the opposite sign in the reworked code.

In [ ]:
@case('systems')
def Ncomp_two_stage_compression():
    return run(ComponentModels.Ncomp('n', media.Air(), n=1.0, Xi=AIR, T_in=400.0, P_in=1.0, N_stage=2,
                                      eff_comp=[0.7, 0.7], P_comp=[2.0, 3.0], T_HT=[350.0, 350.0]))

@case('systems')
def SimpleCompressorSys():
    return run(ComponentModels.SimpleCompressorSys('s', 1.0, AIR, 300.0, 1.0, 5.0))

@case('systems')
def SimpleHeaterSys():
    return run(ComponentModels.SimpleHeaterSys('s', 1.0, AIR, 300.0, 1.0, 800.0))

@case('systems')
def SOCBasicSys_electrolysis():
    return run(SOCsystem.SOCBasicSys('s', 1.0, None, 1.0, None, -50.0, 100))

@case('systems')
def SOCsysRecupe_electrolysis():
    return run(SOCsystem.SOCsysRecupe('s', I=-50.0))

@case('systems')
def SOCsysRecupe_fuel_cell():
    return run(SOCsystem.SOCsysRecupe('s', nf0=1.0, na0=2.0, I=50.0, Ncells=500))

@case('systems')
def SOECSys_electrolysis():
    return run(ElectBox.SOECSys('b', I=-50.0, Ncells=100))

@case('systems')
def SOECSysV2_electrolysis():
    return run(ElectBox.SOECSysV2('b', I=-50.0, Ncells=100))

@case('systems')
def CO2Sys_co_electrolysis_box():
    return run(ElectBox.CO2Sys('b', I=-50.0, Ncells=100))

@case('systems')
def SOECSys_pressurised_1073K_5bar():
    return run(ElectBox.SOECSys('b', I=-80.0, Ncells=100, P_soc=5.0, P0=5.0))

@case('systems')
def AELSys_hydrogen_plant():
    return run(ElectBox.AELSys('b', n=1.0, Sep=np.array([0, 1, 0, -1])))
summary('systems')

## 7 Fischer-Tropsch module

Fails to import or run in the original code (reference composition overwritten); recorded as errors there.

In [ ]:
try:
    from celeste.FT import componentsFT, flowFT, heatFT, interfacesFT, mediaFT, reactorsFT
    _ft_ok = True
except Exception as e:
    _ft_ok = False
    RESULTS['FT_import'] = dict(group='FT', status='error', values={}, success=None, time_s=0.0, errors={'model': f'{type(e).__name__}: {e}'[:300]})

if _ft_ok:
    def _ftprop(name, T, P, phase):
        @case('FT', f'FT_props_{name}_{T:g}K_{P:g}bar_{phase}')
        def _():
            return interfacesFT.Flange('f', getattr(mediaFT, name)(), T=T, P=P, phase=phase)
    for _name in ['Syncrude', 'Wax', 'Naphtha', 'Kerosene', 'Distillate', 'LightGas', 'Water', 'Hydrogen']:
        _ftprop(_name, 450.0, 5.0, 'Gas')
    for _name in ['Naphtha', 'Wax', 'Water']:
        _ftprop(_name, 320.0, 1.0, 'Liquid')

    @case('FT')
    def FT_reactor():
        x = np.zeros(55); x[[0, 1, 2, 4]] = 0.62, 0.05, 0.30, 0.03
        return run(reactorsFT.FT('r', alpha=0.85, CO_conv=0.6, F0_Xi=x))

    @case('FT')
    def FT_reactor_alpha_09():
        x = np.zeros(55); x[[0, 1, 2, 4]] = 0.62, 0.05, 0.30, 0.03
        return run(reactorsFT.FT('r', alpha=0.9, CO_conv=0.7, F0_Xi=x))

    @case('FT')
    def FT_hydrocracker():
        return run(reactorsFT.Hydrocracker('h', mediaFT.Wax()))

    @case('FT')
    def FT_compressor():
        return run(componentsFT.CompIsent('c', mediaFT.LightGas(), eff=0.8))

    @case('FT')
    def FT_heater():
        h = heatFT.Heater('ht', mediaFT.Naphtha(), mediaFT.Naphtha(), F1_T=400.0)
        return run(h)

    @case('FT')
    def FT_gpu_2s_light_gas_split():
        return run(componentsFT.GPU_2S('g', mediaFT.Syncrude()))
summary('FT')

## 8 Save the snapshot

In [ ]:
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
snapshot = dict(label=LABEL, celeste=str(Path(celeste.__file__).parent), shims=SHIMS, cases=RESULTS)
SNAPSHOT_FILE = SNAPSHOT_DIR / f'{LABEL}.json'
SNAPSHOT_FILE.write_text(json.dumps(snapshot, indent=1, sort_keys=True))
print('saved', SNAPSHOT_FILE, f'({len(RESULTS)} cases, {sum(len(r["values"]) for r in RESULTS.values())} values)')
s = summary()
print(s.groupby(['group', 'status']).size().unstack(fill_value=0))
s[s.status != 'ok']

## 9 Compare two snapshots

Set `FILE_A` (the reference, e.g. the original code) and `FILE_B` (the version under review). Values are compared with a
relative tolerance `RTOL` and an absolute tolerance `ATOL`; `ATOL` is in the unit of the quantity (K, bar, mol/s, kJ/s, V, ...).

In [ ]:
FILE_A = SNAPSHOT_DIR / 'main.json'        # reference
FILE_B = SNAPSHOT_FILE                     # this run; set to any other snapshot to compare two stored ones
RTOL, ATOL = 1e-6, 1e-8  # ATOL 1e-8 hides ~1e-9 noise on trace-species LHV

In [ ]:
def load(path):
    return json.loads(Path(path).read_text())

def compare(a, b, rtol=RTOL, atol=ATOL):
    """One row per (case, quantity) present in either snapshot."""
    rows = []
    for case_name in sorted(set(a['cases']) | set(b['cases'])):
        ra, rb = a['cases'].get(case_name), b['cases'].get(case_name)
        if ra is None or rb is None:
            rows.append(dict(case=case_name, quantity='(case)', A=None, B=None, status='only in B' if ra is None else 'only in A'))
            continue
        if ra['status'] != rb['status']:
            rows.append(dict(case=case_name, quantity='(status)', A=ra['status'] + ': ' + ra['errors'].get('model', '')[:60],
                             B=rb['status'] + ': ' + rb['errors'].get('model', '')[:60], status='status'))
        for q in sorted(set(ra['values']) | set(rb['values'])):
            va, vb = ra['values'].get(q), rb['values'].get(q)
            if va is None or vb is None:
                rows.append(dict(case=case_name, quantity=q, A=va, B=vb, status='only in A' if vb is None else 'only in B'))
                continue
            d = vb - va
            rel = abs(d)/max(abs(va), abs(vb), 1e-300)
            rows.append(dict(case=case_name, quantity=q, A=va, B=vb, diff=d, rel=rel,
                             status='same' if abs(d) <= atol + rtol*max(abs(va), abs(vb)) else 'changed'))
    return pd.DataFrame(rows)

A, B = load(FILE_A), load(FILE_B)
print(f"A = {A['label']}   B = {B['label']}")
print('shims A:', A.get('shims') or 'none', '| shims B:', B.get('shims') or 'none')
cmp = compare(A, B)
cmp['group'] = cmp['case'].map({k: r['group'] for k, r in {**A['cases'], **B['cases']}.items()})

In [ ]:
# one line per case: what changed
def case_summary(cmp):
    g = cmp[cmp.quantity != '(case)']
    out = []
    for name, d in cmp.groupby('case', sort=False):
        ra, rb = A['cases'].get(name), B['cases'].get(name)
        ch = d[d.status == 'changed']
        out.append(dict(case=name, group=d.group.iloc[0],
                        A=ra['status'] if ra else '-', B=rb['status'] if rb else '-',
                        solver_A=ra['success'] if ra else None, solver_B=rb['success'] if rb else None,
                        values=int((d.status.isin(['same', 'changed'])).sum()), changed=len(ch),
                        max_rel=float(ch.rel.max()) if len(ch) else 0.0,
                        time_A=round(ra['time_s'], 3) if ra else None, time_B=round(rb['time_s'], 3) if rb else None,
                        only_one_side=int(d.status.isin(['only in A', 'only in B']).sum())))
    return pd.DataFrame(out).set_index('case')

overview = case_summary(cmp)
print(overview.groupby('group').agg(cases=('A', 'size'), unchanged=('changed', lambda s: int((s == 0).sum())),
                                    with_changes=('changed', lambda s: int((s > 0).sum()))))
overview

In [ ]:
# cases with differences, largest first
overview[(overview.changed > 0) | (overview.A != overview.B) | (overview.only_one_side > 0)].sort_values('max_rel', ascending=False)

In [ ]:
# the changed quantities of one case (set CASE) or of all cases, ranked by relative difference
CASE = None
d = cmp[cmp.status == 'changed']
if CASE:
    d = d[d.case == CASE]
d.sort_values('rel', ascending=False)[['case', 'quantity', 'A', 'B', 'diff', 'rel']].head(60)

In [ ]:
# headline quantities side by side: heat, work, exergy destruction, entropy generation, voltages
headline = ('.Q', '.W', '.Exloss', '.Sgen', '.Vop', '.Vid', 'tot_Exloss', 'EBcheck', 'MBcheck')
h = cmp[cmp.quantity.str.endswith(headline) & cmp.status.isin(['same', 'changed'])]
h[h.status == 'changed'].sort_values('rel', ascending=False)[['case', 'quantity', 'A', 'B', 'rel']].head(60)

In [ ]:
# timing: solve time per case, A against B (seconds)
t = overview[['time_A', 'time_B']].dropna()
t['B/A'] = (t.time_B/t.time_A).round(2)
t.sort_values('time_A', ascending=False).head(20)